# Build knowledge graph for datasets from dynamical.org catalog


https://dynamical.org/catalog/


In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import json
from datetime import UTC, datetime, timedelta
from textwrap import dedent
from typing import Literal

import xarray as xr
from e84_geoai_common.llm.core.llm import LLM, LLMInferenceConfig, LLMMessage
from e84_geoai_common.llm.extraction import extract_data_from_text
from e84_geoai_common.llm.models.claude import CLAUDE_4_6_OPUS, BedrockClaudeLLM
from pydantic import AliasChoices, BaseModel, Field, RootModel
from rich import print as rprint  # pyright: ignore[reportUnusedImport]  # noqa: F401
from rich.pretty import pretty_repr
from tqdm import tqdm

from llm_agent.utils.knowledge_graph_construction.graph_construction import build_graph, save_graph
from llm_agent.utils.knowledge_graph_construction.metadata_parsing import (
    Dataset,
    Dimension,
    Variable,
)


In [ ]:
# If True, create only a single global "canonical" instance of each dim without
# the noisy long_name and units metadata fields.
# If False, create an instance for each occurence of a dim i.e. there will be
# a unique time dim for each dataset. These will also include noisy metadata
# fields.
USE_CANONICAL_DIMS = True

assert USE_CANONICAL_DIMS, "Only USE_CANONICAL_DIMS=True is supported currently."

In [ ]:
uris = {
    "forecast": [
        "https://data.dynamical.org/noaa/gfs/forecast/latest.zarr",
        "https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr",
        "https://data.dynamical.org/noaa/hrrr/forecast-48-hour/latest.zarr",
        "https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr",
    ],
    "historical": [
        "https://data.dynamical.org/noaa/gfs/analysis-hourly/latest.zarr",
        "https://data.dynamical.org/noaa/gefs/analysis/latest.zarr",
        "https://data.dynamical.org/noaa/hrrr/analysis/latest.zarr",
    ],
}

---


## Custom overrides


In [ ]:
dim_std_names = {
    "init_time": "forecast_reference_time",
    "lead_time": "forecast_period",
    "ensemble_member": "realization",
}

In [ ]:
dim_alt_names = {
    "latitude": "lat",
    "longitude": "lon",
    "init_time": "time",
}

In [ ]:
overrides = {
    "https://data.dynamical.org/noaa/gfs/forecast/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gfs-forecast/",
    },
    "https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gefs-forecast-35-day",
    },
    "https://data.dynamical.org/noaa/hrrr/forecast-48-hour/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-hrrr-forecast-48-hour",
    },
    "https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/ecmwf-ifs-ens-forecast-15-day-0-25-degree",
    },
    "https://data.dynamical.org/noaa/gfs/analysis-hourly/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gfs-analysis-hourly/",
    },
    "https://data.dynamical.org/noaa/gefs/analysis/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gefs-analysis",
    },
    "https://data.dynamical.org/noaa/hrrr/analysis/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-hrrr-analysis",
    },
}

## Parse metadata


In [ ]:
# Dynamical.org metadata models


class DatasetAttrs(BaseModel):
    dataset_id: str = Field(validation_alias=AliasChoices("dataset_id", "id"))
    dataset_version: str | None = None
    name: str
    description: str
    attribution: str
    spatial_domain: str
    spatial_resolution: str
    time_domain: str
    time_resolution: str
    forecast_domain: str | None = None
    forecast_resolution: str | None = None


class DataVarAttrs(BaseModel):
    long_name: str
    short_name: str | None = None
    standard_name: str | None = None
    units: str
    comment: str | None = None
    step_type: Literal["instant", "accum", "avg", "min", "max"] | None = None
    ensemble_statistic: str | None = None

In [ ]:
class HistoricalDataset(Dataset):
    pass


class ForecastDataset(Dataset):
    lead_times: list[timedelta]

In [ ]:
class HistoricalDatasetTemporalInfo(BaseModel):
    start_date: datetime
    end_date: datetime | None
    temporal_resolution: timedelta | None


class ForecastDatasetTemporalInfo(BaseModel):
    start_date: datetime
    end_date: datetime | None
    temporal_resolution: timedelta | None
    lead_times: list[timedelta]


class DatasetTemporalInfo(RootModel[HistoricalDatasetTemporalInfo | ForecastDatasetTemporalInfo]):
    pass

In [ ]:
def ds_to_prompt(ds: xr.Dataset) -> str:
    """Convert an xarray Dataset to a string prompt for LLM input."""
    out = dedent(f"""
        Dataset Metadata:
        ```json
        {json.dumps(ds.attrs)}
        ```

        Dataset Dimensions:
        ```json
        {json.dumps(list(ds.dims))}
        ```
    """)
    if "lead_time" in ds:
        out += dedent(f"""
            `lead_time` data:
            ```
            {pretty_repr(ds["lead_time"])}
            ```
        """)
    return out

In [ ]:
def extract_temporal_info_from_ds(ds: xr.Dataset) -> DatasetTemporalInfo:
    return extract_data_from_text(
        llm=BedrockClaudeLLM(model_id=CLAUDE_4_6_OPUS),
        model_type=DatasetTemporalInfo,
        system_prompt=dedent(f"""
            Extract the temporal information from the given dataset metadata into the output schema below.

            Output Schema:
            ```json
            {DatasetTemporalInfo.model_json_schema()}
            ```
            """),  # noqa: E501
        user_prompt=ds_to_prompt(ds),
    )

In [ ]:
def extract_dataset_metadata(uri: str) -> Dataset:
    ds = xr.open_zarr(uri, chunks="auto")  # pyright: ignore[reportUnknownMemberType]
    ds_metadata = DatasetAttrs.model_validate(ds.attrs)
    vars_metadata: dict[str, DataVarAttrs] = {
        var_name: DataVarAttrs.model_validate(ds[var_name].attrs) for var_name in ds.data_vars
    }  # pyright: ignore[reportAssignmentType]
    dim_names: list[str] = list(ds.dims)  # pyright: ignore[reportAssignmentType]
    dims_out = {
        dim_name: Dimension(
            id=dim_name,
            standard_name=dim_std_names.get(dim_name, dim_name),
            field_name=dim_name,
            alternate_name=dim_alt_names.get(dim_name),
            long_name=ds[dim_name].attrs.get("long_name"),
            units=ds[dim_name].attrs.get("units"),
        )
        for dim_name in dim_names
    }
    vars_out = [
        Variable(
            id=f"{var_name}_{ds_metadata.dataset_id}",
            standard_name=var_metadata.standard_name,
            field_name=var_name,
            long_name=var_metadata.long_name,
            units=var_metadata.units,
            dims=[dims_out[dim_name] for dim_name in ds[var_name].dims],  # pyright: ignore[reportArgumentType]
        )
        for var_name, var_metadata in vars_metadata.items()
    ]

    temporal_info = extract_temporal_info_from_ds(ds)
    if temporal_info.root.end_date is None:
        # use today's date
        temporal_info.root.end_date = datetime.now(tz=UTC)

    if isinstance(temporal_info.root, ForecastDatasetTemporalInfo):
        description_extra_items = ds_metadata.model_dump(
            include={
                "spatial_domain",
                "spatial_resolution",
                "forecast_domain",
                "forecast_resolution",
                "attribution",
            }
        )
        description_extra = "\n".join(f"{k}: {v}" for k, v in description_extra_items.items())
        expanded_description = f"{ds_metadata.description}\n\n{description_extra}"
        out = ForecastDataset(
            id=ds_metadata.dataset_id,
            title=ds_metadata.name,
            summary=expanded_description,
            uri=uri,
            temporal_resolution=temporal_info.root.temporal_resolution,
            start_date=temporal_info.root.start_date.replace(tzinfo=UTC),
            end_date=temporal_info.root.end_date.replace(tzinfo=UTC),
            lead_times=temporal_info.root.lead_times,
            variables=vars_out,
            keywords=[],
            dims=list(dims_out.values()),
            metadata_link=overrides.get(uri, {}).get("landing_page"),
        )
    else:
        description_extra_items = ds_metadata.model_dump(
            include={
                "spatial_domain",
                "spatial_resolution",
                "attribution",
            }
        )
        description_extra = "\n".join(f"{k}: {v}" for k, v in description_extra_items.items())
        expanded_description = f"{ds_metadata.description}\n\n{description_extra}"
        out = HistoricalDataset(
            id=ds_metadata.dataset_id,
            title=ds_metadata.name,
            summary=expanded_description,
            uri=uri,
            temporal_resolution=temporal_info.root.temporal_resolution,
            start_date=temporal_info.root.start_date.replace(tzinfo=UTC),
            end_date=temporal_info.root.end_date.replace(tzinfo=UTC),
            variables=vars_out,
            keywords=[],
            dims=list(dims_out.values()),
            metadata_link=overrides.get(uri, {}).get("landing_page"),
        )
    return out

In [ ]:
datasets: dict[str, list[Dataset]] = {
    "historical": [],
    "forecast": [],
}

In [ ]:
with tqdm(uris["historical"]) as bar:
    datasets["historical"] = [extract_dataset_metadata(uri) for uri in bar]

In [ ]:
with tqdm(uris["forecast"]) as bar:
    datasets["forecast"] = [extract_dataset_metadata(uri) for uri in bar]

---


## Populate graph


In [ ]:
# Exclude the GEFS forecast dataset. There's something off with the chunking of this dataset.
# The lat/lon coordinate sorting step we do as a preprocessing step in the data analysis tool
# stalls/takes too long.
forecast_datasets_minus_gefs = [ds for ds in datasets["forecast"] if "gefs" not in ds.id]

g = build_graph(
    datasets["historical"] + forecast_datasets_minus_gefs,
    canonical_dims=True,
    media_type="application/zarr",
)

## Save


In [ ]:
out_path = "../../src/llm_agent/data/dataset-kg-dynamical-org-generated.ttl"
save_graph(g, out_path)